# U1-04 · MNIST: regresión logística y primera red en TensorFlow

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 1, apartados 1.4 a 1.7 y 1.10*

En Colab, activa la GPU si quieres ir más rápido: *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Sin ella funciona igual, algo más despacio.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges, con licencia Creative Commons Reconocimiento-CompartirIgual 3.0.

## 1. Explorar MNIST

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
print("entrenamiento:", x_ent.shape, x_ent.dtype,
      " prueba:", x_pru.shape)
print("valores de los píxeles:", x_ent.min(), "a", x_ent.max())
print("imágenes por dígito:", np.bincount(y_ent))
print(f"píxeles con tinta: {(x_ent > 0).mean():.1%}")
nunca = (x_ent.max(axis=0) == 0).sum()
print("píxeles siempre en blanco:", nunca, "de 784")

fig, ejes = plt.subplots(2, 10, figsize=(4.4, 1.0))
for d in range(10):
    ejes[0, d].imshow(x_ent[np.flatnonzero(y_ent == d)[0]])
    ejes[1, d].imshow(x_ent[y_ent == d].mean(axis=0))
    ejes[0, d].set_title(str(d), fontsize=7, loc="center")
for eje in ejes.flat:
    eje.axis("off")
fig.savefig("mnist.pdf")
plt.show()

## 2. Agrupar sin etiquetas (k-medias)

In [ ]:
import numpy as np
import keras
from sklearn.cluster import KMeans

_, (x, y) = keras.datasets.mnist.load_data()  # 10 000 imágenes
X = x.reshape(-1, 784) / 255.0

# no supervisado: se dan las imágenes, pero no las etiquetas
km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(X)
aciertos, vistos = 0, set()
for g in range(10):
    en_grupo = y[km.labels_ == g]
    mayoritario = np.bincount(en_grupo).argmax()
    aciertos += (en_grupo == mayoritario).sum()
    vistos.add(mayoritario)
    print(f"grupo {g}: {len(en_grupo):5d} imágenes, "
          f"{np.mean(en_grupo == mayoritario):.0%} "
          f"son un {mayoritario}")
print("dígitos sin grupo propio:", sorted(set(range(10)) - vistos))
print(f"si cada grupo se etiqueta con su dígito mayoritario: "
      f"{aciertos / len(y):.1%} de acierto")

## 3. Regresión logística: el modelo de referencia

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
from sklearn.linear_model import LogisticRegression

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
X_ent = x_ent.reshape(-1, 784) / 255.0   # 784 variables entre 0 y 1
X_pru = x_pru.reshape(-1, 784) / 255.0

# 1) modelo estadístico: ¿qué píxeles distinguen un 0 de un 1?
sel = y_ent <= 1
m01 = LogisticRegression(max_iter=1000).fit(X_ent[sel], y_ent[sel])
sel_p = y_pru <= 1
print(f"0 frente a 1: {m01.score(X_pru[sel_p], y_pru[sel_p]):.4f}")
plt.figure(figsize=(2.6, 2.2))
plt.imshow(m01.coef_.reshape(28, 28), cmap="RdBu_r",
           vmin=-1, vmax=1)
plt.colorbar(label="coeficiente (+: más probable un 1)")
plt.axis("off")
plt.savefig("coeficientes_01.pdf")

# 2) modelo de referencia para los diez dígitos
t0 = time.perf_counter()
m = LogisticRegression(max_iter=1000).fit(X_ent, y_ent)
print(f"10 dígitos: {m.score(X_pru, y_pru):.4f} de acierto en prueba"
      f" ({time.perf_counter() - t0:.0f} s, "
      f"{m.coef_.size + m.intercept_.size} parámetros)")
plt.show()

## 4. ¿Quién aprovecha más los datos?

Tarda unos minutos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras
from sklearn.linear_model import LogisticRegression

keras.utils.set_random_seed(0)
(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
X_ent = x_ent.reshape(-1, 784) / 255.0
X_pru = x_pru.reshape(-1, 784) / 255.0

tamanos = [1000, 3000, 10_000, 30_000, 60_000]
logistica, red = [], []
for n in tamanos:
    m = LogisticRegression(max_iter=1000).fit(X_ent[:n], y_ent[:n])
    logistica.append(m.score(X_pru, y_pru))
    r = keras.Sequential([
        keras.Input(shape=(784,)),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])
    r.compile("adam", "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    r.fit(X_ent[:n], y_ent[:n], epochs=5, verbose=0)
    red.append(r.evaluate(X_pru, y_pru, verbose=0)[1])
    print(f"{n:6d} imágenes: logística {logistica[-1]:.4f}, "
          f"red {red[-1]:.4f}")

plt.plot(tamanos, logistica, "o-", label="regresión logística")
plt.plot(tamanos, red, "o-", label="red con 128 neuronas ocultas")
plt.xscale("log")
plt.xlabel("imágenes de entrenamiento (escala logarítmica)")
plt.ylabel("acierto en prueba")
plt.legend()
plt.savefig("mas_datos.pdf")
plt.show()

## 5. La primera red con TensorFlow

In [ ]:
import time
import numpy as np
import keras

keras.utils.set_random_seed(0)              # resultados repetibles
(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()
x_ent, x_pru = x_ent / 255.0, x_pru / 255.0  # píxeles entre 0 y 1

def red(ocultas):
    capas = [keras.Input(shape=(28, 28)), keras.layers.Flatten()]
    capas += [keras.layers.Dense(n, activation="relu")
              for n in ocultas]
    capas += [keras.layers.Dense(10, activation="softmax")]
    modelo = keras.Sequential(capas)
    modelo.compile(optimizer="adam",
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])
    return modelo

for ocultas in ([], [128]):           # sin capa oculta y con una
    modelo = red(ocultas)
    t0 = time.perf_counter()
    h = modelo.fit(x_ent, y_ent, epochs=5, batch_size=32,
                   validation_split=0.1, verbose=0)
    seg = time.perf_counter() - t0
    perdida, acierto = modelo.evaluate(x_pru, y_pru, verbose=0)
    print(f"capas ocultas {ocultas}: {modelo.count_params():,} "
          f"parámetros, {seg:.0f} s, validación "
          f"{h.history['val_accuracy'][-1]:.4f}, "
          f"prueba {acierto:.4f}")

prob = modelo.predict(x_pru, verbose=0)
pred = prob.argmax(axis=1)
fallos = np.flatnonzero(pred != y_pru)
print(len(fallos), "fallos de 10 000; por época, validación:",
      np.round(h.history["val_accuracy"], 4))

import matplotlib.pyplot as plt
epocas = range(1, 6)
plt.plot(epocas, h.history["accuracy"], "o-", label="entrenamiento")
plt.plot(epocas, h.history["val_accuracy"], "o-", label="validación")
plt.xlabel("época"); plt.ylabel("acierto"); plt.xticks(epocas)
plt.legend()
plt.savefig("curvas.pdf")
print("entrenamiento por época:", np.round(h.history["accuracy"], 4))

fig, ejes = plt.subplots(1, 8, figsize=(4.4, 0.9))
for eje, i in zip(ejes, fallos[:8]):
    eje.imshow(x_pru[i])
    eje.set_title(f"{y_pru[i]} → {pred[i]}\n{prob[i].max():.0%}",
                  fontsize=6, loc="center", fontweight="normal")
    eje.axis("off")
fig.savefig("fallos.pdf")
pares = np.bincount(y_pru[fallos] * 10 + pred[fallos], minlength=100)
for c in np.argsort(pares)[::-1][:3]:
    print(f"real {c // 10} tomado por {c % 10}: {pares[c]} veces")
plt.show()

**Prueba.** Cambia la capa oculta a 32 y a 512 neuronas, y las épocas a 10. Anota parámetros, tiempo y acierto en prueba.

> Deng, L. (2012). The MNIST database of handwritten digit images for machine learning research. *IEEE Signal Processing Magazine*, 29(6), 141–142. https://doi.org/10.1109/MSP.2012.2211477
>
> Chollet, F. (2021). *Deep Learning with Python* (2.ª ed.). Manning. ISBN 978-1-61729-686-4